# PSX LSTM Predictor — Colab Runner
End-to-end pipeline: data download → features → training → predictions → plots.

**Runtime:** GPU recommended (`Runtime > Change runtime type > T4 GPU`)


In [ ]:
# ── 1. Clone repository ───────────────────────────────────────────────
REPO_URL  = "https://github.com/BasitMujtaba/psx-lstm-predictor.git"
REPO_NAME = "psx-lstm-predictor"

import os
if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL}
else:
    %cd /content/{REPO_NAME}
    !git pull origin main

%cd /content/{REPO_NAME}
print("✓ Repo ready")


In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────
!pip install -q \
    yfinance \
    transformers==4.40.0 \
    feedparser==6.0.11 \
    python-dateutil \
    scikit-learn \
    torch \
    tqdm \
    pyyaml \
    nbformat \
    matplotlib

print("✓ Dependencies installed")


In [ ]:
# ── 3. Verify GPU ────────────────────────────────────────────────────
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU found — training will be slow on CPU.")


In [ ]:
# ── 4. Load and inspect config ───────────────────────────────────────
import yaml, os
os.chdir(f"/content/{REPO_NAME}")

with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

print(f"Date range : {cfg['data']['start_date']}  →  {cfg['data']['end_date']}")
print(f"Tickers    : {len(cfg['data']['tickers'])} tickers")
print(f"Model      : {cfg['model']['architecture'].upper()}")
print(f"Seq len    : {cfg['training']['seq_len']}")
print(f"Epochs     : {cfg['training']['max_epochs']}")
print(f"Batch size : {cfg['training']['batch_size']}")


In [ ]:
# ── 5. Download PSX OHLCV data ───────────────────────────────────────
import sys
sys.path.insert(0, f"/content/{REPO_NAME}")

# clear cached module to ensure latest version is used
for mod in list(sys.modules.keys()):
    if "psx_downloader" in mod:
        del sys.modules[mod]

from src.data_collection.psx_downloader import run as dl_run

prices_df = dl_run(cfg)

print(f"\n✓ Shape     : {prices_df.shape}")
print(f"  Tickers   : {prices_df['ticker'].nunique()}")
print(f"  Date range: {prices_df['date'].min().date()} → {prices_df['date'].max().date()}")
print(f"\n  Rows per ticker:")
print(prices_df.groupby('ticker').size().to_string())


## News Sentiment  *(optional — skip if you want faster runs)*
Run the next cell to collect and score news with FinBERT.
Skip it to use zero sentiment (all sentiment features = 0).


In [ ]:
# ── 6. Collect and score news sentiment ─────────────────────────────
# Comment out this cell to skip news and use zero sentiment instead.
import pandas as pd
from src.data_collection.news_scraper import run as news_run

trading_dates = pd.DatetimeIndex(sorted(prices_df['date'].unique()))

sentiment_df = news_run(cfg=cfg, trading_dates=trading_dates)
print(f"\n✓ Sentiment rows: {len(sentiment_df)}")
print(sentiment_df[["date", "sentiment_score",
                     "article_count"]].tail(5).to_string(index=False))


In [ ]:
# ── 6b. (Alternative) Load cached sentiment or use zero sentiment ────
# Uncomment this cell if you skipped the news cell above.

# import pandas as pd, os
# sent_path = os.path.join(cfg["data"]["raw_news_dir"],
#                          "news_sentiment_daily.csv")
# if os.path.exists(sent_path):
#     sentiment_df = pd.read_csv(sent_path, parse_dates=["date"])
#     print(f"Cached sentiment loaded: {len(sentiment_df)} rows")
# else:
#     sentiment_df = None
#     print("No cached sentiment — using zero sentiment.")


In [ ]:
# ── 7. Compute indicators and build features ─────────────────────────
import pandas as pd, os
from src.feature_engineering.indicators import add_all_indicators
from src.feature_engineering.features   import (
    build_features, FEATURE_COLS, TARGET_COL,
)

proc_dir = cfg["data"]["processed_dir"]
os.makedirs(proc_dir, exist_ok=True)

features_dict = {}

for ticker, df_raw in prices_df.groupby('ticker'):
    df_raw = df_raw.copy().reset_index(drop=True)
    df_raw.columns = [c.lower() for c in df_raw.columns]
    df_raw = df_raw[["date", "open", "high", "low", "close", "volume"]].dropna()
    df_raw["date"] = pd.to_datetime(df_raw["date"])

    df_ind  = add_all_indicators(df_raw, cfg=cfg)
    df_feat = build_features(df_ind, sentiment_df=sentiment_df, cfg=cfg)

    out_path = os.path.join(proc_dir, f"{ticker}_features.csv")
    df_feat.to_csv(out_path, index=False)
    features_dict[ticker] = df_feat
    print(f"  {ticker:12s}  {len(df_feat):>5d} rows  "
          f"{len(FEATURE_COLS)} features")

print(f"\n✓ Features built for {len(features_dict)} tickers")
print(f"  Feature cols : {FEATURE_COLS}")
print(f"  Target col   : {TARGET_COL}")


In [ ]:
# ── 8. Scale features (per-ticker, train-only fit) ───────────────────
from src.models.scaler import run as scale_run

scaled_dict, scaler_dict = scale_run(
    features_dict = features_dict,
    feature_cols  = FEATURE_COLS,
    target_col    = TARGET_COL,
    out_dir       = cfg["data"]["processed_dir"],
    train_ratio   = cfg["training"].get("train_ratio", 0.70),
    val_ratio     = cfg["training"].get("val_ratio",   0.15),
)

print(f"\n✓ Scaling complete for {len(scaled_dict)} tickers")
for ticker, (tr, va, te) in scaled_dict.items():
    print(f"  {ticker:12s}  train={len(tr):>4d}  val={len(va):>4d}  test={len(te):>4d}")


In [ ]:
# ── 9. Build DataLoaders ─────────────────────────────────────────────
from src.models.dataset import make_loaders

train_loader, val_loader, test_loader = make_loaders(
    scaled_dict  = scaled_dict,
    feature_cols = FEATURE_COLS,
    target_col   = TARGET_COL,
    seq_len      = cfg["training"]["seq_len"],
    batch_size   = cfg["training"]["batch_size"],
    num_workers  = 0,
    pin_memory   = (device == "cuda"),
)

print(f"\n✓ DataLoaders ready")
print(f"  train batches : {len(train_loader)}")
print(f"  val   batches : {len(val_loader)}")
print(f"  test  batches : {len(test_loader)}")

import torch
X_sample, y_sample = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  X shape : {X_sample.shape}   (batch, seq_len, features)")
print(f"  y shape : {y_sample.shape}   (batch,)")


In [ ]:
# ── 10. Build model ──────────────────────────────────────────────────
from src.models.lstm import build_model

model = build_model(cfg)
model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✓ Model built: {cfg['model']['architecture'].upper()}")
print(f"  Trainable parameters: {n_params:,}")
print(model)


In [ ]:
# ── 11. Train ────────────────────────────────────────────────────────
import os
from src.models.trainer import train as train_model

ckpt_dir  = cfg["training"]["checkpoint_dir"]
ckpt_path = os.path.join(ckpt_dir, "ALL_TICKERS_best.pt")
os.makedirs(ckpt_dir, exist_ok=True)

history = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    cfg          = cfg,
    ckpt_path    = ckpt_path,
    feature_cols = FEATURE_COLS,
    target_col   = TARGET_COL,
)

print(f"\n✓ Training complete")
print(f"  Best val loss : {min(history['val_loss']):.6f}")
print(f"  Epochs run    : {len(history['train_loss'])}")


In [ ]:
# ── 12. Plot training history ────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history["train_loss"], label="Train loss", color="#4CAF50")
plt.plot(history["val_loss"],   label="Val loss",   color="#FF9800",
         linestyle="--")
plt.axvline(history["val_loss"].index(min(history["val_loss"])),
            color="#B0BEC5", linestyle=":", label="Best epoch")
plt.title("Training History", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (scaled)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── 13. (Optional) Train one model per ticker ────────────────────────
# from src.models.trainer import run as train_run
#
# history_dict = train_run(
#     scaled_dict  = scaled_dict,
#     feature_cols = FEATURE_COLS,
#     target_col   = TARGET_COL,
#     cfg          = cfg,
#     ckpt_dir     = cfg["training"]["checkpoint_dir"],
# )
# print(f"✓ Trained {len(history_dict)} models")


In [ ]:
# ── 14. Generate predictions ─────────────────────────────────────────
from src.evaluation.predict import build_prediction_df, compute_metrics
import os

pred_dir = cfg["data"]["predictions_dir"]
os.makedirs(pred_dir, exist_ok=True)

pred_dict    = {}
metrics_dict = {}

for ticker in scaled_dict:
    print(f"Predicting: {ticker}")
    pred_df = build_prediction_df(
        ticker       = ticker,
        scaled_dict  = scaled_dict,
        scaler_dict  = scaler_dict,
        cfg          = cfg,
        model        = model,
    )
    pred_df.to_csv(
        os.path.join(pred_dir, f"{ticker}_predictions.csv"), index=False
    )
    metrics = compute_metrics(pred_df, split="test")
    pred_dict[ticker]    = pred_df
    metrics_dict[ticker] = metrics

print("\n=== Test Metrics ===")
for ticker, m in metrics_dict.items():
    print(f"  {ticker:12s}  "
          f"MAE={m.get('MAE', 0):.5f}  "
          f"RMSE={m.get('RMSE', 0):.5f}  "
          f"DirAcc={m.get('DirectionalAcc_pct', 0):.1f}%")


In [ ]:
# ── 15. Generate and display plots ───────────────────────────────────
cfg["plots"] = cfg.get("plots", {})
cfg["plots"]["show"] = True

from src.evaluation.plots import (
    plot_prediction_vs_actual,
    plot_scatter,
    plot_residuals,
    plot_directional_accuracy,
    plot_metrics_summary,
)

for ticker, pred_df in pred_dict.items():
    print(f"\n── {ticker} ──")
    plot_prediction_vs_actual(pred_df, ticker, pred_dir, show=True)
    plot_scatter(pred_df,   ticker, pred_dir, show=True)
    plot_residuals(pred_df, ticker, pred_dir, show=True)

plot_directional_accuracy(metrics_dict, pred_dir, show=True)
plot_metrics_summary(metrics_dict,      pred_dir, show=True)


In [ ]:
# ── 16. (Optional) Copy outputs to Google Drive ──────────────────────
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# dest = "/content/drive/MyDrive/psx-lstm-outputs"
# shutil.copytree(
#     f"/content/{REPO_NAME}/data",
#     dest,
#     dirs_exist_ok=True,
# )
# print(f"✓ Outputs copied to Google Drive: {dest}")


In [ ]:
# ── 17. (Alternative) Run full pipeline.py in one shot ───────────────
# %cd /content/{REPO_NAME}
# !python pipeline.py
# !python pipeline.py --skip-news
# !python pipeline.py --skip-news --skip-train
